# Classification models with Scikit-learn

- Binary classification: $y_i \in \{0, 1\}$
- Predictors: $\mathbf{x}_i$ as before

**Models studied in this notebook:**

- Logistic regression: linear and polynomial features
- Random forest

***
## Example data for classification

- Two features $(x_{1i}, x_{2i})$; binary outcome $y_i$
- Nonlinear relationship with noise:
  $$
  \begin{aligned}
  y_i &=
  \begin{cases}
      1 & \text{if } f(x_{1i}, x_{2i}) + \epsilon_i \geq 0 \\
      0 & \text{otherwise}
  \end{cases} \\
  f(x_{1i}, x_{2i}) &= \sin(2\pi x_{1i}) \cos(\pi x_{2i}) \\
  \epsilon_i &\stackrel{\text{iid}}{\sim}
      \mathcal{N}(0, \sigma_{\epsilon}^2)
  \end{aligned}
  $$

### Creating a demo dataset

In [ ]:
import numpy as np


def f(x1, x2):
    """
    True function for classification examples
    """
    return np.sin(2 * np.pi * x1) * np.cos(np.pi * x2)

In [ ]:
def create_class_data(N=100, sigma=0.2, rng=None):
    """
    Create synthetic data for binary classification examples.

    Parameters
    ----------
    N : int
        Number of observations to generate.
    sigma : float
        Standard deviation of the error term.
    rng : numpy.random.Generator, optional
        Random number generator to use.

    Returns
    -------
    X : array-like
        Feature matrix.
    y : array-like
        Binary response variable.

    """

    if rng is None:
        rng = np.random.default_rng(seed=1234)

    # Draw features from uniform distribution
    x1 = rng.uniform(0, 1, size=N)
    x2 = rng.uniform(0, 1, size=N)

    z = f(x1, x2)

    # Add noise to latent variable if noise variance is positive
    if sigma > 0:
        # Draw errors from normal distribution
        epsilon = rng.normal(0, sigma, size=N)
        z += epsilon

    # Convert latent variable to binary response variable with values 0 and 1
    y = (z >= 0).astype(int)

    # Stack features into matrix
    X = np.column_stack((x1, x2))

    return X, y

In [ ]:
# Sample size
N = 100

# Standard deviation of noise
sigma_eps = 0.2

# Create demo dataset for classification
X, y = create_class_data(N=N, sigma=sigma_eps)

### Inspecting the sample

- Class labels and counts
- Scatter plot with different markers for each class
    - Plotting helpers: imports from `lecture4_plots.py`

In [ ]:
import pandas as pd

# TODO: Tabulate class labels and counts

In [ ]:
from lecture4_plots import plot_classes

# Plot demo dataset for classification
_ = plot_classes(X, y)

***
### Training and test samples

- Fixed dataset: reserve test observations before fitting or tuning
- Class proportions: comparison across training and test samples
- Split with
[`train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
from sklearn.model_selection import train_test_split

# Fraction of the dataset to be used for testing
test_size = 0.3

X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X, y, test_size=test_size, random_state=123
)

print('Distribution of classes in demo training sample:')
print(pd.Series(y_train_demo).value_counts() / len(y_train_demo))

print('\nDistribution of classes in demo test sample:')
print(pd.Series(y_test_demo).value_counts() / len(y_test_demo))

# TODO: Preserve class proportions with stratification

### Training sample for this example

- All 100 synthetic observations for training
- Fresh test sample of 10,000 observations after fitting and tuning

In [ ]:
# Use the complete synthetic sample for training
X_train = X
y_train = y

***
## Logistic regression

- Probability of class 1:
  $$
  p(\mathbf{x}_i) =
  \frac{1}{1 + \exp[-(\mu + \mathbf{x}_i'\bm{\beta})]}
  $$
- Sigmoid function: $\sigma(z) = 1/(1 + e^{-z})$
  - Linear index: $z_i = \mu + \mathbf{x}_i'\bm{\beta}$
  - Any real index mapped into $(0, 1)$

In [ ]:
from lecture4_plots import plot_sigmoid

_ = plot_sigmoid()

### Loss and regularization

- Negative log-likelihood plus penalty:
  $$
  L(\mu, \bm{\beta}) =
  -\underbrace{\frac{1}{N}\mathcal{L}(\mu,\bm{\beta})}
      _{\text{scaled log-likelihood}}
  +\underbrace{\frac{r(\bm{\beta})}{NC}}_{\text{regularization}}
  $$
- $r(\bm\beta)$: L1, L2, elastic net, or no penalty
- Larger $C$ implies smaller penalty
- $C = \infty$: no regularization

***
### Fitting a simple model

- Linear index with two features:
  $$
  z_i = \mu + \beta_1 x_{1i} + \beta_2 x_{2i}
  $$
- No regularization: `C=np.inf`
- No scaling needed for these two inputs on $[0, 1]$

#### Step 1: Define and fit the model

- Estimator:
[`LogisticRegression()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- `C`: inverse regularization strength
- `l1_ratio`: penalty type
- `max_iter`: maximum number of solver iterations

| Penalty | `l1_ratio` | `C` |
|---------|------------|-----|
| L1 | 1 | finite |
| L2 | 0 | finite |
| Elastic net | between 0 and 1 | finite |
| None | 0 | `np.inf` |

- Default solver `'lbfgs'`: L2 or no penalty
- Solver `'saga'`: also L1 and elastic net

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: Define unpenalized logistic regression with an intercept and max_iter=1000
# lr = ...

# TODO: Fit lr on X_train and y_train

#### Step 2: Inspect decision boundaries

- Supplied plotting functions from `lecture4_plots.py`

In [ ]:
from lecture4_plots import plot_decision_boundary

# Create x-values to evaluate decisions (used for either dimension)
xvalues = np.linspace(0, 1, 1000)

ax = plot_classes(X_train, y_train)
plot_decision_boundary(ax, xvalues, lr)
_ = ax.set_title('Classification with logistic regression')

#### Step 3: Compute training metrics

- Confusion matrix: observed versus predicted class labels
- True positives (TP), true negatives (TN), false positives (FP), false negatives (FN)

In [ ]:
from lecture4_plots import plot_generic_confusion_matrix
from sklearn.metrics import accuracy_score

plot_generic_confusion_matrix()

- Accuracy: share of correct predictions,
[`accuracy_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
  $$
  ACC = \frac{TP + TN}{FP + FN + TP + TN}
  $$
- Precision: share of true positives among predicted positives,
[`precision_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
  $$
  PRE = \frac{TP}{TP + FP}
  $$
- Recall: share of detected positives,
[`recall_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
  $$
  REC = \frac{TP}{FN + TP}
  $$
- F1 score: harmonic mean of precision and recall,
[`f1_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
  $$
  F1 = 2\frac{PRE \cdot REC}{PRE + REC}
  $$

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score


def tabulate_training_metrics(y_train, y_train_pred):
    """
    Print accuracy, precision, recall, and F1 score for training predictions.

    Parameters
    ----------
    y_train : array
        Observed class labels in the training sample.
    y_train_pred : array
        Predicted class labels for the training observations.
    """
    acc_train = accuracy_score(y_train, y_train_pred)
    pre = precision_score(y_train, y_train_pred)
    rec = recall_score(y_train, y_train_pred)
    f1 = f1_score(y_train, y_train_pred)

    print(f'Training accuracy:  {acc_train:.3f}')
    print(f'Training precision: {pre:.3f}')
    print(f'Training recall:    {rec:.3f}')
    print(f'Training F1 score:  {f1:.3f}')

In [ ]:
# TODO: Predict class labels for the training observations with lr
# y_train_pred = ...

# TODO: Print all four training metrics with tabulate_training_metrics()
# Use y_train and y_train_pred

#### Step 4: Plot the confusion matrix

- Observed and predicted training labels
- Plot with
[`ConfusionMatrixDisplay.from_predictions()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Plot confusion matrix for the training predictions
_ = ConfusionMatrixDisplay.from_predictions(
    y_train,
    y_train_pred,
    colorbar=False,
    cmap='Blues',
    text_kw={'fontsize': 12, 'fontweight': 'bold'},
).ax_.set_title('Confusion matrix for linear index (training)')

***
### Fitting a model with polynomials

- Polynomial terms and interactions:
  $$
  \begin{aligned}
  z_i ={}& \mu + \beta_1 x_{1i} + \beta_2 x_{2i}
      + \beta_3 x_{1i}x_{2i} \\
      &+ \beta_4 x_{1i}^2 + \beta_5 x_{2i}^2 + \dots
  \end{aligned}
  $$
- More flexible decision boundaries

#### Step 1: Define and fit the pipeline

- Polynomial degree: 5
- Pipeline: scaling, polynomial features, scaling, logistic regression
- No bias column in `PolynomialFeatures`; intercept in `LogisticRegression`
- No regularization: `C=np.inf`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

degree = 5

# TODO: Define unpenalized logistic regression with an intercept and max_iter=1_000
# lr = ...

# TODO: Build pipeline: StandardScaler, PolynomialFeatures, StandardScaler, lr
# Use degree=degree and include_bias=False for PolynomialFeatures
# pipe_lr = ...

# TODO: Fit pipe_lr on X_train and y_train

#### Step 2: Inspect decision boundaries

In [ ]:
# Create x-values used to evaluate decisions
xvalues = np.linspace(0, 1, 1000)
ax = plot_classes(X_train, y_train)
plot_decision_boundary(ax, xvalues, pipe_lr)
_ = ax.set_title('Classification with logistic regression (polynomials)')

#### Step 3: Compute training metrics

- Accuracy, precision, recall, and F1 score

In [ ]:
# TODO: Predict class labels for the training observations with pipe_lr
# y_train_pred = pipe_lr.predict(X_train)

# TODO: Tabulate training metrics for polynomial logistic regression
# tabulate_training_metrics(y_train, y_train_pred)

#### Step 4: Plot the confusion matrix

- Training predictions for the polynomial model
- Perfect training fit: possible overfitting

In [ ]:
# Plot confusion matrix for the training predictions
_ = ConfusionMatrixDisplay.from_predictions(
    y_train,
    y_train_pred,
    colorbar=False,
    cmap='Blues',
    text_kw={'fontsize': 12, 'fontweight': 'bold'},
).ax_.set_title('Confusion matrix for polynomial features (training)')

***
### Cross-validating the penalty term

- Joint selection of inverse penalty strength `C` and polynomial degree
- Cross-validation of the complete pipeline with
[`GridSearchCV()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
    - Scaling fitted separately within each training fold
    - Scaling fitted before cross-validation: data leakage

#### Step 1: Define folds and scoring

- Five shuffled, reproducible folds with
[`StratifiedKFold()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
    - Stratification: preservation of proportions, including any imbalance
    - Approximately equal class proportions across folds
- Accuracy: approximately balanced classes; equal costs of classification errors

In [ ]:
from sklearn.model_selection import StratifiedKFold

# TODO: Define five stratified folds with shuffle=True and random_state=1234
# cv = ...

# Scoring rule for all model searches
scoring = 'accuracy'

#### Step 2: Define the pipeline

- Scaling, polynomial features, scaling, L2 logistic regression
- L2 penalty: `l1_ratio=0.0`; `max_iter=1_000`

In [ ]:
# TODO: Build the pipeline with both scalers and L2 logistic regression
# Use include_bias=False for PolynomialFeatures
# pipe_lr = ...

#### Step 3: Define and run the grid search

- 100 values of $C$: log scale from $10^{-2}$ to $10^2$
- Polynomial degrees: 1 through 5
- Pipeline parameters:
  - `logisticregression__C`
  - `polynomialfeatures__degree`
- `refit=True`: selected pipeline refitted on all training observations

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: Define 100 candidate values of C with np.logspace(-2, 2, 100)
# c_values = ...

# TODO: Define candidate polynomial degrees 1 through 5
# degrees = ...

# TODO: Define the grid with polynomialfeatures__degree and logisticregression__C
# param_grid = ...

# TODO: Define GridSearchCV for cross-validating C and polynomial degree
# lr_cv = ...

# TODO: Fit lr_cv on the original X_train and y_train

#### Step 4: Inspect the selected model

- Selected `C` and degree: `best_params_`
- Mean validation accuracy: `best_score_`
- Predictions and plots: directly through `lr_cv`

In [ ]:
# TODO: Report the selected C and polynomial degree from lr_cv.best_params_

# TODO: Report mean validation accuracy from lr_cv.best_score_

#### Step 5: Inspect decision boundaries

In [ ]:
# Create x-values used to evaluate decisions
xvalues = np.linspace(0, 1, 1000)
ax = plot_classes(X_train, y_train)
plot_decision_boundary(ax, xvalues, lr_cv)
_ = ax.set_title('Classification with logistic regression (CV)')

#### Step 6: Compute training metrics

- Independent test comparison after the random forest section

In [ ]:
# TODO: Predict training labels with lr_cv
# y_train_pred = lr_cv.predict(X_train)

# TODO: Tabulate training metrics for cross-validated logistic regression
# tabulate_training_metrics(y_train, y_train_pred)

***
### Validation curves

- Training and validation accuracy against `C`
- Polynomial degree fixed at 4 for this illustration
- Same stratified folds and accuracy scoring as the grid search

#### Step 1: Define a separate pipeline

- Steps: Scaling, degree-4 polynomial features, scaling, L2 logistic regression

In [ ]:
# TODO: Build a separate pipeline with degree=4 and include_bias=False
# Use both scalers and LogisticRegression(l1_ratio=0.0, max_iter=1_000)
# pipe_lr_validation = ...

#### Step 2: Compute training and validation scores

- Computation with
[`validation_curve()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html)
    - `param_name`: pipeline step and parameter joined by `__`
    - `param_range`: candidate values of `C`
- Returns `test_scores`: validation folds within the training sample

In [ ]:
from sklearn.model_selection import validation_curve

# TODO: Define 100 values of C with np.logspace(-4, 4, 100)
# param_range = ...

param_name = 'logisticregression__C'

# TODO: Compute validation_curve for pipe_lr_validation on X_train and y_train
# train_scores, test_scores = ...

#### Step 3: Plot the validation curve

- Supplied function from `lecture4_plots.py`
- Lines: mean accuracy across folds
- Shaded bands: one standard deviation above and below the mean

In [ ]:
from lecture4_plots import plot_accuracy_validation_curve

plot_accuracy_validation_curve(
    param_range,
    train_scores,
    test_scores,
    score='Accuracy',
    title='Validation curve for logistic regression (polynomial degree 4)',
)

***
## Random forest

- Ensemble: average class probabilities from many trees
- Nonlinear relationships without polynomial expansion or standardization

### Step 1: Define and fit the forest

- Estimator:
[`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
    - `n_estimators`: number of trees
    - `max_depth`: maximum tree depth
    - `max_features`: candidate features per split
    - `min_samples_leaf`: minimum observations in each leaf
    - `min_samples_split`: minimum observations before considering a split
    - `random_state`: reproducible sampling and feature selection
    - `n_jobs`: parallel jobs

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: Define a forest with n_estimators=100 and max_depth=3
# Use random_state=1234 and n_jobs=4
# forest = ...

# TODO: Fit forest on the original X_train and y_train

### Step 2: Inspect decision boundaries

In [ ]:
ax = plot_classes(X_train, y_train)
plot_decision_boundary(ax, xvalues, forest)
_ = ax.set_title('Classification with random forest')

### Step 3: Compute training metrics

In [ ]:
# TODO: Predict training labels with forest
# y_train_pred_forest = forest.predict(X_train)

# TODO: Tabulate training metrics for random forest
# tabulate_training_metrics(y_train, y_train_pred_forest)

### Cross-validating random forest hyperparameters

- Joint selection of `max_depth` and `n_estimators`
- Ignore the other hyperparameters for this illustration

#### Step 1: Define and run the grid search

- Fresh forest estimator with `random_state=1234` and `n_jobs=1`
- Parallel grid search with `n_jobs=4`
- Original, unscaled features

In [ ]:
# TODO: Define a fresh forest with random_state=1234 and n_jobs=1
# forest_tune = ...

# TODO: Define depths 5 through 15
# max_depth_grid = ...

# TODO: Define tree counts
# n_estimators_grid = ...

# TODO: Define the grid with max_depth and n_estimators
# param_grid = ...

# TODO: Define GridSearchCV for forest_tune with the shared cv and scoring
# forest_cv = ...

# TODO: Fit forest_cv on X_train and y_train

In [ ]:
print(f'Best validation accuracy: {forest_cv.best_score_:.3f}')
for param, value in forest_cv.best_params_.items():
    print(f'Best {param}: {value}')

#### Step 2: Inspect decision boundaries

In [ ]:
ax = plot_classes(X_train, y_train)
plot_decision_boundary(ax, xvalues, forest_cv)
_ = ax.set_title('Classification with random forest (CV)')

#### Step 3: Compute training metrics

In [ ]:
# TODO: Predict training labels with forest_cv
# y_train_pred_forest_cv = forest_cv.predict(X_train)

# TODO: Tabulate training metrics for cross-validated random forest
# tabulate_training_metrics(y_train, y_train_pred_forest_cv)

***
## Comparing predictions on new observations

- Fresh test sample: 10,000 observations; same noise level
- Separate random generator with seed 5678

In [ ]:
# Generate a large independent test sample after fitting all main models
rng_test = np.random.default_rng(seed=5678)
X_test, y_test = create_class_data(N=10_000, sigma=sigma_eps, rng=rng_test)

### Training and test comparison

- `CV ACC`: mean validation accuracy; training table only
- Training and test metrics: final models refitted on all training observations

In [ ]:
# Compare models selected by cross-validation
models = {'Logistic, CV': lr_cv, 'Random forest, CV': forest_cv}

# Report the same four metrics for the training and test samples
samples = {'Training': (X_train, y_train), 'Test': (X_test, y_test)}

for sample_name, (X_sample, y_sample) in samples.items():
    print(f'\n{sample_name} sample:')
    header = (
        f'{"Model":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} '
        f'{"F1 score":>10}'
    )
    if sample_name == 'Training':
        header += f' {"CV ACC":>10}'
    print(header)
    for name, model in models.items():
        y_sample_pred = model.predict(X_sample)
        acc = accuracy_score(y_sample, y_sample_pred)
        pre = precision_score(y_sample, y_sample_pred)
        rec = recall_score(y_sample, y_sample_pred)
        f1 = f1_score(y_sample, y_sample_pred)

        row = f'{name:<25} {acc:10.3f} {pre:10.3f} {rec:10.3f} {f1:10.3f}'
        if sample_name == 'Training':
            cv_text = f'{model.best_score_:.3f}'
            row += f' {cv_text:>10}'
        print(row)